<a href="https://colab.research.google.com/github/NateMophi/NLP-453-Coursework/blob/Code/NLP_F1_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
f1_df = pd.read_csv("/content/drive/MyDrive/F1/F1_tweets.csv", engine='python', on_bad_lines="skip", )

In [ ]:
f1_df.shape
sample_tweet = f1_df['text'].iloc[67]

# APPROACH A **(ROBERTA)**

In [ ]:
from transformers import pipeline
sentiment = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")



In [ ]:
print(f"The sentiment results are as follows: {sentiment(sample_tweet)}")

# APPROACH B **(VaDER)**

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download("vader_lexicon")
sia = SentimentIntensityAnalyzer()


In [ ]:
scores = sia.polarity_scores(sample_tweet)
print(f"Tweet Scores of '{sample_tweet}'\n {scores}")

In [ ]:
def get_score(tweet):
  return sia.polarity_scores(tweet)['compound']

In [ ]:
f1_df["sentiment_score"] = f1_df["text"].apply(get_score)

In [ ]:
from collections import Counter

all_words = ' '.join(f1_df['text']).split()
word_freq = Counter(all_words)

# ranked by frequency
all_ranked = word_freq.most_common(1000)

# Known F1 drivers and teams (2021-2022 grid)
known_drivers = [
    'Hamilton', 'Lewis', 'Max', 'Verstappen', 'Leclerc', 'Charles',
    'Sainz', 'Carlos', 'Norris', 'Lando', "Russell", "George",
    'Alonso', 'Fernando', 'Stroll', 'Lance', 'Ocon', 'Esteban',
    'Albon', 'Alexander', 'Gasly', 'Pierre', 'Magnussen', 'Kevin',
    'Ricciardo', 'Daniel', 'Perez', 'Sergio', 'Bottas', 'Valtteri',
    'Zhou', 'Guanyu', 'Tsunoda', 'Yuki', 'Hulkenberg', "Nico", "Schumacher",
    "Mick", "Nicolas", "Latifi"
]

known_teams = [
    'Mercedes', 'Ferrari', 'Red Bull', "RB", 'McLaren',
    'Aston Martin', "Aston", 'Alfa Romeo', 'Haas', 'Alpine', 'Williams',
    "Alpha Tauri", "Sauber"
]

print("="*80)
print("DRIVERS FOUND IN DATASET:")
print("="*80)

found_drivers = {}
for driver in known_drivers:
    for word, count in all_ranked:
        if word.lower() == driver.lower():
            found_drivers[driver] = count
            print(f"{driver:20s} - {count:6d} occurrences")

print("\n" + "="*80)
print("TEAMS FOUND IN DATASET:")
print("="*80)

found_teams = {}
for team in known_teams:
    for word, count in all_ranked:
        if word.lower() == team.lower():
            found_teams[team] = count
            print(f"{team:20s} - {count:6d} occurrences")

# Entity masking list
print("\n" + "="*80)
print("SORTED BY FREQUENCY (for entity masking):")
print("="*80)

print("\nDrivers (most to least frequent):")
for driver, count in sorted(found_drivers.items(), key=lambda x: x[1], reverse=True):
    print(f"  '{driver.lower()}',  # {count} occurrences")

print("\nTeams (most to least frequent):")
for team, count in sorted(found_teams.items(), key=lambda x: x[1], reverse=True):
    print(f"  '{team.lower()}',  # {count} occurrences")

In [ ]:
import re
import string
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

nltk.download("stopwords")
from nltk.corpus import stopwords



# Preprocessing

In [ ]:
def min_process(text):
# keep URLs, emojis, mentions, hastags

  if pd.isna(text):
    return ""
  return str(text)

def standard_process(text):
# remove URLs, punctuations, mentions, emojis
# lowercasing

  if pd.isna(text):
    return ""

  text = str(text)
# Lowercase
  text = text.lower()

# URLs
  text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

#  Mentions
  text = re.sub(r'@\w+', '', text)

# Emojis
  text = re.sub(r'[^\w\s#]', '', text, flags=re.UNICODE)

# Whitespace
  text = ' '.join(text.split())

  return text


def entity_process(text):
  """
    F1-Aware + Aggressive pre-processing
    - Everything from strategy_standard
    - F1-specific entity masking
    - Replace team/driver names with generic tokens
    """
  if pd.isna(text):
    return ""

  text = str(text)

  text = str(text)


  text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

  text = re.sub(r'@\w+', '<USER>', text)

  text = re.sub(r'[^\w\s#<>]', '', text, flags=re.UNICODE)

  text = text.lower()

  # F1-SPECIFIC ENTITY MASKING
  # f1_entities = {
  #       r'\bmercedes\b': '<TEAM>',
  #       r'\bsilver arrows\b': '<TEAM>',
  #       r'\bscuderia\b': '<TEAM>',
  #       r'\balpha tauri\b': '<TEAM>',
  #       r'\bred bull\b': '<TEAM>',
  #       r'\bferrari\b': '<TEAM>',
  #       r'\bmclaren\b': '<TEAM>',
  #       r'\baston martin\b': '<TEAM>',
  #       r'\balfa romeo\b': '<TEAM>',
  #       r'\bhaas\b': '<TEAM>',
  #       r'\balpine\b': '<TEAM>',
  #       r'\bwilliams\b': '<TEAM>',

  #       r'\bgoatifi\b': '<DRIVER>',
  #       r'\bLH\b': '<DRIVER>',
  #       r'\bmax\b': '<DRIVER>',
  #       r'\bverstappen\b': '<DRIVER>',
  #       r'\bcharles\b': '<DRIVER>',
  #       r'\bstroll\b': '<DRIVER>',
  #       r'\blance\b': '<DRIVER>',
  #       r'\bgasly\b': '<DRIVER>',
  #       r'\bpierre\b': '<DRIVER>',
  #       r'\bbottas\b': '<DRIVER>',
  #       r'\bvalterri\b': '<DRIVER>',
  #       r'\balonso\b': '<DRIVER>',
  #       r'\bricciardo\b': '<DRIVER>',
  #       r'\bcheco\b': '<DRIVER>',
  #       r'\bperez\b': '<DRIVER>',
  #       r'\bsergio\b': '<DRIVER>',
  #       r'\besteban\b': '<DRIVER>',
  #       r'\bocon\b': '<DRIVER>',
  #       r'\byuki\b': '<DRIVER>',
  #       r'\btsunoda\b': '<DRIVER>',
  #       r'\bgeorge\b': '<DRIVER>',
  #       r'\brussell\b': '<DRIVER>',
  #       r'\bgiovinazzi\b': '<DRIVER>',
  #       r'\bantonio\b': '<DRIVER>',
  #       r'\bmick\b': '<DRIVER>',
  #       r'\bschumacher\b': '<DRIVER>',
  #       r'\bnicolas\b': '<DRIVER>',
  #       r'\bfernando\b': '<DRIVER>',
  #       r'\biceman\b': '<DRIVER>',
  #       r'\kimi\b': '<DRIVER>',
  #       r'\braikkonen\b': '<DRIVER>',
  #       r'\bkubica\b': '<DRIVER>',
  #       r'\brobert\b': '<DRIVER>',
  #       r'\blatifi\b': '<DRIVER>',
  #       r'\bhamilton\b': '<DRIVER>',
  #       r'\blewis\b': '<DRIVER>',
  #       r'\bverstappen\b': '<DRIVER>',
  #       r'\bleclerc\b': '<DRIVER>',
  #       r'\bseb\b': '<DRIVER>',
  #       r'\bnorris\b': '<DRIVER>',
  #       r'\blando\b': '<DRIVER>',
  #       r'\bvettel\b': '<DRIVER>',
  #       r'\bsainz\b': '<DRIVER>',
  #       r'\bcarlos\b': '<DRIVER>',
  #   }
  teams = ['mercedes', 'merc', 'silver arrows', 'scuderia', 'red bull', 'ferrari',
             'mclaren', 'alpha tauri', 'aston martin', 'alfa romeo', 'haas', 'alpine', 'williams']
  for team in teams:
        text = text.replace(team, ' TEAM ')

  drivers = [
    'goatifi', 'LH', 'max', 'verstappen', 'charles', 'stroll',
    'lance', 'gasly', 'pierre', 'bottas', 'valterri', 'alonso',
    'ricciardo', 'checo', 'perez', 'sergio', 'esteban', 'ocon',
    'yuki', 'tsunoda', 'george', 'russell', 'giovinazzi', 'antonio',
    'mick', 'schumacher', 'nicolas', 'fernando', 'iceman', 'kimi',
    'raikkonen', 'kubica', 'robert', 'latifi', 'hamilton', 'lewis',
    'leclerc', 'seb', 'norris', 'lando', 'vettel', 'sainz', 'carlos'
  ]
  for driver in drivers:
        text = re.sub(r'\b' + driver + r'\b', ' DRIVER ', text)
  # Remove extra whitespace
  text = ' '.join(text.split())

  return text

# Tokenization **(sub-word Level)**

In [ ]:
def tokenise_word(text):
  return word_tokenize(text.lower())

def tokenise_sentence(text):
  return sent_tokenize(text.lower())

def tokenise_subword(text):
  """
    Simple subword tokenization (character-level n-grams)
    Using character bigrams as approximation of subword tokens
    """
  text = text.replace(' ', '') # remove spaces for char lvl
  bigrams = [text[i:i+2] for i in range(len(text) - 1)]
  return bigrams


In [ ]:
print("Applying preprocessing to over 600K tweets...")


f1_df['text_minimal'] = f1_df['text'].apply(min_process)
f1_df['text_standard'] = f1_df['text'].apply(standard_process)
f1_df['text_f1_aware'] = f1_df['text'].apply(entity_process)


print("Preprocessing complete!")


In [ ]:
# Sampling 1K tweets for detailed analysis
sample_size = 1000
sample_df = f1_df.sample(n=min(sample_size, len(f1_df)), random_state=69)

print(f"\nApplying tokenization to {len(sample_df)} sampled tweets...")

In [ ]:
# Word level tokenisation on each strategy
sample_df['tokens_minimal_word'] = sample_df['text_minimal'].apply(tokenise_word)
sample_df['tokens_standard_word'] = sample_df['text_standard'].apply(tokenise_word)
sample_df['tokens_f1aware_word'] = sample_df['text_f1_aware'].apply(tokenise_word)

In [ ]:
# Example Inspection
print("\n" + "="*80)
print("PREPROCESSING COMPARISON - Example Tweet")
print("="*80)

idx = 7
print(f"\nOriginal:\n  {f1_df['text'].iloc[idx]}\n")
print(f"Strategy A (Minimal):\n  {f1_df['text_minimal'].iloc[idx]}\n")
print(f"Strategy B (Standard):\n  {f1_df['text_standard'].iloc[idx]}\n")
print(f"Strategy C (F1-Aware):\n  {f1_df['text_f1_aware'].iloc[idx]}\n")

print("="*80)
print("TOKENIZATION COMPARISON - Example from Sample")
print("="*80)

idx_sample = 9
print(f"\nWord tokens (Minimal):\n  {sample_df['tokens_minimal_word'].iloc[idx_sample]}\n")
print(f"Word tokens (Standard):\n  {sample_df['tokens_standard_word'].iloc[idx_sample]}\n")
print(f"Word tokens (F1-Aware):\n  {sample_df['tokens_f1aware_word'].iloc[idx_sample]}\n")

In [ ]:
print("Re-running sentiment analysis on preprocessed text...\n")

f1_df['sentiment_minimal'] = f1_df['text_minimal'].apply(lambda x: sia.polarity_scores(x)['compound'])
f1_df['sentiment_standard'] = f1_df['text_standard'].apply(lambda x: sia.polarity_scores(x)['compound'])
f1_df['sentiment_f1aware'] = f1_df['text_f1_aware'].apply(lambda x: sia.polarity_scores(x)['compound'])

print("Re-analysis complete!\n")


In [ ]:
print("="*80)
print("SENTIMENT SCORE COMPARISON")
print("="*80)

comparison = pd.DataFrame({
    'Minimal': f1_df['sentiment_minimal'].describe(),
    'Standard': f1_df['sentiment_standard'].describe(),
    'F1-Aware': f1_df['sentiment_f1aware'].describe(),
})

print(comparison)

# Calculate differences
print("\n" + "="*80)
print("DIVERGENCE ANALYSIS: How much do preprocessing strategies differ?")
print("="*80)

In [ ]:
diff_minimal_standard = (f1_df['sentiment_minimal'] - f1_df['sentiment_standard']).abs().mean()
diff_minimal_f1aware = (f1_df['sentiment_minimal'] - f1_df['sentiment_f1aware']).abs().mean()
diff_standard_f1aware = (f1_df['sentiment_standard'] - f1_df['sentiment_f1aware']).abs().mean()

print(f"Average difference (Minimal vs Standard): {diff_minimal_standard:.4f}")
print(f"Average difference (Minimal vs F1-Aware): {diff_minimal_f1aware:.4f}")
print(f"Average difference (Standard vs F1-Aware): {diff_standard_f1aware:.4f}")

In [ ]:
print("TOP 5 DIVERGENCE EXAMPLES (Standard vs F1-Aware)")
print("="*80)

f1_df['divergence'] = (f1_df['sentiment_standard'] - f1_df['sentiment_f1aware']).abs()
top_diverge = f1_df.nlargest(5, 'divergence')[['text', 'sentiment_standard', 'sentiment_f1aware', 'divergence']]

for idx, row in top_diverge.iterrows():
    print(f"\nTweet: {row['text'][:80]}...")
    print(f"  Standard: {row['sentiment_standard']:.4f}")
    print(f"  F1-Aware: {row['sentiment_f1aware']:.4f}")
    print(f"  Difference: {row['divergence']:.4f}")


In [ ]:
if 'sentiment_standard' not in sample_df.columns:
    sample_df = sample_df.copy()
    sample_df['sentiment_standard'] = f1_df.loc[sample_df.index, 'sentiment_standard']
    sample_df['sentiment_minimal'] = f1_df.loc[sample_df.index, 'sentiment_minimal']
    sample_df['sentiment_f1aware'] = f1_df.loc[sample_df.index, 'sentiment_f1aware']

    print("Sentiment columns added to sample_df")
else:
    print("Sentiment columns already in sample_df")

# Verify
print(f"sample_df shape: {sample_df.shape}")
print(f"Columns: {sample_df.columns.tolist()}")

# Tokenization **(word Level)**

In [ ]:
# TOKEN STATISTICS ANALYSIS
print("Computing token statistics on sample...\n")

# Compute token statistics
sample_df['token_count_minimal'] = sample_df['tokens_minimal_word'].apply(len)
sample_df['token_count_standard'] = sample_df['tokens_standard_word'].apply(len)
sample_df['token_count_f1aware'] = sample_df['tokens_f1aware_word'].apply(len)

# Unique token ratio (diversity of vocabulary)
sample_df['unique_ratio_minimal'] = sample_df['tokens_minimal_word'].apply(
    lambda x: len(set(x)) / len(x) if len(x) > 0 else 0
)

sample_df['unique_ratio_standard'] = sample_df['tokens_standard_word'].apply(
    lambda x: len(set(x)) / len(x) if len(x) > 0 else 0
)

sample_df['unique_ratio_f1aware'] = sample_df['tokens_f1aware_word'].apply(
    lambda x: len(set(x)) / len(x) if len(x) > 0 else 0
)

print("="*80)
print("TOKEN STATISTICS BY STRATEGY")
print("="*80 + "\n")

# Show statistics
token_stats = pd.DataFrame({
    'Minimal_Count': sample_df['token_count_minimal'].describe(),
    'Standard_Count': sample_df['token_count_standard'].describe(),
    'F1Aware_Count': sample_df['token_count_f1aware'].describe(),
})

print("TOKEN COUNT PER TWEET:")
print(token_stats)

print("\n" + "="*80)
print("UNIQUE TOKEN RATIO (Vocabulary Diversity):")
print("="*80 + "\n")

unique_stats = pd.DataFrame({
    'Minimal_Ratio': sample_df['unique_ratio_minimal'].describe(),
    'Standard_Ratio': sample_df['unique_ratio_standard'].describe(),
    'F1Aware_Ratio': sample_df['unique_ratio_f1aware'].describe(),
})

print(unique_stats)

print("\n" + "="*80)
print("INTERPRETATION:")
print("="*80)
print(f"Minimal strategy: {sample_df['token_count_minimal'].mean():.1f} tokens per tweet")
print(f"  (keeping URLs, emojis, mentions increases token count)")
print(f"\nStandard strategy: {sample_df['token_count_standard'].mean():.1f} tokens per tweet")
print(f"  (removing URLs/emojis/mentions reduces token count)")
print(f"\nF1-Aware strategy: {sample_df['token_count_f1aware'].mean():.1f} tokens per tweet")
print(f"  (entity masking has minimal impact on token count)")

# Most common tokens per sentiment
print("\n" + "="*80)
print("MOST COMMON TOKENS BY SENTIMENT CATEGORY")
print("="*80 + "\n")

# Tweets by sentiment
sample_df['sentiment_label'] = sample_df['sentiment_standard'].apply(
    lambda x: 'positive' if x > 0.05 else ('negative' if x < -0.05 else 'neutral')
)

# Get all tokens for positive tweets
positive_tokens = []
for tokens_list in sample_df[sample_df['sentiment_label'] == 'positive']['tokens_standard_word']:
    positive_tokens.extend(tokens_list)

negative_tokens = []
for tokens_list in sample_df[sample_df['sentiment_label'] == 'negative']['tokens_standard_word']:
    negative_tokens.extend(tokens_list)

neutral_tokens = []
for tokens_list in sample_df[sample_df['sentiment_label'] == 'neutral']['tokens_standard_word']:
    neutral_tokens.extend(tokens_list)

# Count frequency
pos_freq = Counter(positive_tokens)
neg_freq = Counter(negative_tokens)
neu_freq = Counter(neutral_tokens)

print("TOP 15 TOKENS IN POSITIVE TWEETS:")
for token, count in pos_freq.most_common(15):
    print(f"  {token:20s} - {count:5d} occurrences")

print("\nTOP 15 TOKENS IN NEGATIVE TWEETS:")
for token, count in neg_freq.most_common(15):
    print(f"  {token:20s} - {count:5d} occurrences")

print("\nTOP 15 TOKENS IN NEUTRAL TWEETS:")
for token, count in neu_freq.most_common(15):
    print(f"  {token:20s} - {count:5d} occurrences")

In [ ]:
import matplotlib.pyplot as plt

# Visualization 1: Sentiment distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

strategies = {
    'Minimal': f1_df['sentiment_minimal'],
    'Standard': f1_df['sentiment_standard'],
    'F1-Aware': f1_df['sentiment_f1aware']
}

for ax, (name, scores) in zip(axes, strategies.items()):
    ax.hist(scores, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_title(f'Sentiment Distribution\n{name} Pre-processing', fontsize=13, fontweight='bold')
    ax.set_xlabel('VADER Compound Score')
    ax.set_ylabel('Number of Tweets')
    ax.axvline(scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {scores.mean():.3f}')
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig('sentiment_strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Visualization saved...")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import numpy as np

print("Converting date column and handling bad values...\n")

# Try to convert to datetime, coerce errors to NaT
f1_df['date'] = pd.to_datetime(f1_df['date'], errors='coerce')

# Check how many NaT values we have
num_bad_dates = f1_df['date'].isna().sum()
print(f"Rows with bad dates: {num_bad_dates:,} out of {len(f1_df):,}")

# Remove rows with bad dates
if num_bad_dates > 0:
    print(f"Removing {num_bad_dates:,} rows with invalid dates...\n")
    f1_df = f1_df.dropna(subset=['date'])
    print(f"Cleaned dataset: {len(f1_df):,} tweets remaining\n")

print(f"Date range: {f1_df['date'].min()} to {f1_df['date'].max()}")
print(f"Total tweets: {len(f1_df):,}")

# ANALYSIS 1: TEMPORAL SENTIMENT TRENDS

In [ ]:
print("TEMPORAL SENTIMENT ANALYSIS: How has F1 sentiment evolved?")
print("="*100 + "\n")

# Group by month and calculate mean sentiment
f1_df['year_month'] = f1_df['date'].dt.to_period('M')

# Use STANDARD sentiment (most appropriate preprocessing due to )
monthly_sentiment = f1_df.groupby('year_month')['sentiment_standard'].agg([
    ('mean', 'mean'),
    ('median', 'median'),
    ('std', 'std'),
    ('count', 'count')
]).reset_index()

# Convert period to timestamp for plotting
monthly_sentiment['date'] = monthly_sentiment['year_month'].dt.to_timestamp()

print("MONTHLY SENTIMENT STATISTICS:")
print(monthly_sentiment)

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(monthly_sentiment['date'], monthly_sentiment['mean'],
        marker='o', linewidth=2, markersize=6, label='Mean Sentiment', color='steelblue')
ax.fill_between(monthly_sentiment['date'],
                monthly_sentiment['mean'] - monthly_sentiment['std'],
                monthly_sentiment['mean'] + monthly_sentiment['std'],
                alpha=0.2, color='steelblue', label='±1 Std Dev')
ax.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5)

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Sentiment Score (VADER)', fontsize=12, fontweight='bold')
ax.set_title('F1 Overall Sentiment Trends Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Format x-axis
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('temporal_sentiment_trends.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTemporal trends visualization saved...")

# ANALYSIS 2: TEAM SENTIMENT COMPARISON

In [ ]:
# TEAM SENTIMENT ANALYSIS: Which teams have best/worst perception?
print("\n" + "="*100)
print("TEAM SENTIMENT ANALYSIS")
print("="*100 + "\n")

# Columns for team mentions
# Use LOWERCASE search on standard preprocessed text

def contains_team(text, team_keywords):
    """Check if text mentions a team"""
    if pd.isna(text):
        return False
    text_lower = str(text).lower()
    return any(keyword in text_lower for keyword in team_keywords)

# Team keywords
teams_to_track = {
    'Mercedes': ["ground effect", 'mercedes', 'merc', 'silver arrows', "w12", "w13", "zero pod", "Hamilton", "Lewis", "LH", "Lewis Hamilton"],
    'Red Bull': ['red bull', "ground effect", 'redbull', 'rb', "rb18", "rb19", 'max verstappen', "Max", "Checo", "Sergio", "Perez"],
    'Ferrari': ['ferrari', 'f1-75', "ground effect", "Charles", "Smooth Operator", "Carlos", "Sainz", "Leclerc", "Carlos Sainz", "Charles Leclerc"]
}

# Binary columns for each team
for team_name, keywords in teams_to_track.items():
    f1_df[f'mentions_{team_name}'] = f1_df['text_standard'].apply(
        lambda x: contains_team(x, keywords)
    )

# Sentiment for each team
team_stats = {}

for team_name in teams_to_track.keys():
    team_tweets = f1_df[f1_df[f'mentions_{team_name}'] == True]

    if len(team_tweets) > 0:
        team_stats[team_name] = {
            'count': len(team_tweets),
            'mean_sentiment': team_tweets['sentiment_standard'].mean(),
            'median_sentiment': team_tweets['sentiment_standard'].median(),
            'std': team_tweets['sentiment_standard'].std(),
            'positive': (team_tweets['sentiment_standard'] > 0.05).sum(),
            'negative': (team_tweets['sentiment_standard'] < -0.05).sum(),
            'neutral': ((team_tweets['sentiment_standard'] >= -0.05) &
                       (team_tweets['sentiment_standard'] <= 0.05)).sum()
        }

print("TEAM SENTIMENT STATISTICS:")
print("-" * 100)

team_comparison = pd.DataFrame(team_stats).T
team_comparison['positive_ratio'] = team_comparison['positive'] / team_comparison['count'] * 100
team_comparison['negative_ratio'] = team_comparison['negative'] / team_comparison['count'] * 100

print(team_comparison)

# Sort by mean sentiment
print("\n" + "="*100)
print("TEAMS RANKED BY SENTIMENT (Most Positive → Most Negative):")
print("="*100)

sorted_teams = sorted(team_stats.items(), key=lambda x: x[1]['mean_sentiment'], reverse=True)
for rank, (team, stats) in enumerate(sorted_teams, 1):
    # Ratios from the stats
    positive_ratio = (stats['positive'] / stats['count']) * 100
    negative_ratio = (stats['negative'] / stats['count']) * 100

    print(f"\n{rank}. {team}")
    print(f"   Mean Sentiment: {stats['mean_sentiment']:>7.4f}")
    print(f"   Positive: {positive_ratio:>6.1f}% | Negative: {negative_ratio:>6.1f}%")
    print(f"   Tweets: {stats['count']:>6,.0f}")

# ANALYSIS 3: TEAM SENTIMENT OVER TIME

In [ ]:
# TEAM SENTIMENT TRENDS OVER TIME
print("\n" + "="*100)
print("TEAM SENTIMENT TRENDS OVER TIME")
print("="*100 + "\n")

# Grouping by month and team
team_temporal = {}

for team_name in teams_to_track.keys():
    team_monthly = f1_df[f1_df[f'mentions_{team_name}'] == True].groupby('year_month').agg({
        'sentiment_standard': ['mean', 'count']
    }).reset_index()

    team_monthly.columns = ['year_month', 'mean_sentiment', 'tweet_count']
    team_monthly['date'] = team_monthly['year_month'].dt.to_timestamp()

    team_temporal[team_name] = team_monthly

    print(f"{team_name}: {len(team_monthly)} months with mentions, "
          f"{team_monthly['tweet_count'].sum():,} total tweets")

# Multi-line plot
fig, ax = plt.subplots(figsize=(16, 7))

colors = {'Mercedes': '#00D2BE', 'Red Bull': '#0600EF', 'Ferrari': '#DC0000'}

for team_name, data in team_temporal.items():
    ax.plot(data['date'], data['mean_sentiment'],
            marker='o', linewidth=2.5, markersize=8, label=team_name,
            color=colors.get(team_name, 'gray'), alpha=0.8)

ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.3)
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Sentiment Score', fontsize=12, fontweight='bold')
ax.set_title('F1 Team Sentiment Trends Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, loc='best')
ax.grid(alpha=0.3)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('team_sentiment_trends.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n Team sentiment trends visualization saved...")

# PIVOT POINT DETECTION

In [ ]:
# PIVOT POINT DETECTION: When did sentiment shift dramatically?
print("\n" + "="*100)
print("Identifying Major Sentiment Shifts...")

def detect_pivot_points(monthly_data, team_name, window=3, threshold=0.1):
    """
    Detect pivot points
    Args:
        monthly_data: DataFrame with date and sentiment columns
        team_name: Name of team
        window: Number of months for smoothing
        threshold: Minimum sentiment change to count as pivot

    Returns:
        List of pivot point dates and descriptions
    """
    data = monthly_data.sort_values('date').copy()

    data['rolling_mean'] = data['mean_sentiment'].rolling(window=window, center=True).mean()
    data['sentiment_change'] = data['mean_sentiment'].diff()

    # Significant changes
    pivot_points = []
    for idx, row in data.iterrows():
        if pd.notna(row['sentiment_change']) and abs(row['sentiment_change']) > threshold:
            direction = "↑ Improved" if row['sentiment_change'] > 0 else "↓ Declined"
            pivot_points.append({
                'date': row['date'],
                'sentiment': row['mean_sentiment'],
                'change': row['sentiment_change'],
                'direction': direction,
                'tweets': row['tweet_count']
            })

    return pivot_points

# Pivots for each team
print("MAJOR SENTIMENT SHIFTS (change > 0.1 points):")
print("-" * 100)

all_pivots = {}

for team_name, data in team_temporal.items():
    pivots = detect_pivot_points(data, team_name, window=2, threshold=0.1)
    all_pivots[team_name] = pivots

    print(f"\n{team_name}:")

    if pivots:
        for pivot in pivots:
            print(f"  {pivot['date'].strftime('%Y-%m-%d')} - {pivot['direction']}")
            print(f"    Sentiment: {pivot['sentiment']:.4f} (change: {pivot['change']:+.4f})")
            print(f"    Based on {pivot['tweets']:,} tweets\n")
    else:
        print(f"  No major pivot points detected (threshold: 0.1)")

# Highlight pivot points
fig, axes = plt.subplots(len(team_temporal), 1, figsize=(16, 4*len(team_temporal)))

if len(team_temporal) == 1:
    axes = [axes]

colors = {'Mercedes': '#00D2BE', 'Red Bull': '#0600EF', 'Ferrari': '#DC0000'}

for ax, (team_name, data) in zip(axes, team_temporal.items()):
    # Plot line
    ax.plot(data['date'], data['mean_sentiment'],
            marker='o', linewidth=2.5, markersize=8, color=colors.get(team_name, 'gray'), alpha=0.8)

    # Highlight pivot points
    pivots = all_pivots.get(team_name, [])
    for pivot in pivots:
        ax.scatter(pivot['date'], pivot['sentiment'], s=200, zorder=5,
                  color='red' if pivot['change'] < 0 else 'green', marker='*')
        ax.annotate(f"{pivot['direction']}",
                   xy=(pivot['date'], pivot['sentiment']),
                   xytext=(10, 10), textcoords='offset points',
                   fontsize=9, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.7))

    ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.3)
    ax.set_ylabel('Sentiment Score', fontsize=11, fontweight='bold')
    ax.set_title(f'{team_name} - Sentiment with Pivot Points', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)

    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig('pivot_points_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n Pivot point analysis visualization saved...")

# SENTIMENT DISTRIBUTION BY TEAM

In [ ]:
# TEAM SENTIMENT DISTRIBUTION COMPARISON
print("\n" + "="*100)
print("SENTIMENT DISTRIBUTION BY TEAM")
print("="*100 + "\n")

# Comparison plot
fig, axes = plt.subplots(1, len(teams_to_track), figsize=(5*len(teams_to_track), 5))

for ax, team_name in zip(axes, teams_to_track.keys()):
    team_sentiment = f1_df[f1_df[f'mentions_{team_name}'] == True]['sentiment_standard']

    ax.hist(team_sentiment, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(team_sentiment.mean(), color='red', linestyle='--', linewidth=2,
              label=f'Mean: {team_sentiment.mean():.3f}')
    ax.axvline(team_sentiment.median(), color='green', linestyle='--', linewidth=2,
              label=f'Median: {team_sentiment.median():.3f}')

    ax.set_title(f'{team_name}\n(n={len(team_sentiment):,} tweets)',
                fontsize=12, fontweight='bold')
    ax.set_xlabel('Sentiment Score')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('team_sentiment_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("Team sentiment distribution visualization saved!")

# **KEY FINDINGS**

In [ ]:
# SUMMARY: KEY FINDINGS FOR MY RESEARCH QUESTION
print("\n" + "="*100)
print("KEY FINDINGS: F1 SENTIMENT ANALYSIS")
print("="*100)

print("RESEARCH QUESTION:")
print("-" * 100)
print("How has public perception of F1 teams evolved, and can NLP identify pivot points")
print("where sentiment shifted from technical respect to sporting antagonism?")

print("\n" + "="*100)
print("FINDINGS:")
print("="*100)

# Finding 1: Overall Trend
overall_early = f1_df[f1_df['date'] < '2022-01-01']['sentiment_standard'].mean()
overall_late = f1_df[f1_df['date'] >= '2022-01-01']['sentiment_standard'].mean()

print(f"\n1. OVERALL SENTIMENT TREND")
print(f"   Pre-2022:  {overall_early:.4f}")
print(f"   Post-2022: {overall_late:.4f}")
print(f"   Change: {overall_late - overall_early:+.4f}")

# Finding 2: Team Rankings
print(f"\n2. TEAM SENTIMENT RANKINGS (Best to Worst)")
for rank, (team, stats) in enumerate(sorted_teams, 1):
    # Calculate positive_ratio on the fly
    positive_ratio = (stats['positive'] / stats['count']) * 100
    print(f"   {rank}. {team:15s} - {stats['mean_sentiment']:+.4f} ({positive_ratio:.1f}% positive)")

# Finding 3: Volatility
print(f"\n3. SENTIMENT VOLATILITY (Std Dev by Team)")
for team_name in teams_to_track.keys():
    team_tweets = f1_df[f1_df[f'mentions_{team_name}'] == True]
    volatility = team_tweets['sentiment_standard'].std()
    print(f"   {team_name:15s} - {volatility:.4f} (higher = more volatile perception)")

# Finding 4: Pivot Points
print(f"\n4. MAJOR PIVOT POINTS DETECTED")
pivot_count = sum(len(pivots) for pivots in all_pivots.values())
if pivot_count > 0:
    for team_name, pivots in all_pivots.items():
        if pivots:
            print(f"   {team_name}: {len(pivots)} major shifts")
else:
    print(f"   No major pivot points detected with current threshold")

print("\n" + "="*100)